# ETL Transform Notebook

This notebook cleans and enriches the extracted data. It applies business-focused transformations such as deduplication, standardization, enrichment, and KPI calculation so the data is ready for analytics and storage.

## What this notebook does
- Loads the staging files from the extraction step.
- Cleans and normalizes field values.
- Joins related datasets to create richer analytical tables.
- Creates business KPIs for revenue, order value, and feedback quality.
- Writes processed outputs for the loading step.

In [ ]:
# Cell 1 — Load staging datasets
from pathlib import Path
import pandas as pd

STAGING_DIR = Path("ETL/staging")
OUTPUT_DIR = Path("ETL/ETL_processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(STAGING_DIR / "customers.csv")nproducts = pd.read_csv(STAGING_DIR / "products.csv")
orders = pd.read_csv(STAGING_DIR / "orders.csv")
feedback = pd.read_csv(STAGING_DIR / "feedback.csv")
sentiments = pd.read_csv(STAGING_DIR / "sentiments.csv")

print("Loaded staging datasets successfully")
print("Customers shape:", customers.shape)
print("Orders shape:", orders.shape)
print("Feedback shape:", feedback.shape)

In [ ]:
# Cell 2 — Inspect the raw structure before cleaning
print("Customer columns:", customers.columns.tolist())
print("Product columns:", products.columns.tolist())
print("Order columns:", orders.columns.tolist())
print("Feedback columns:", feedback.columns.tolist())
print("Sentiment columns:", sentiments.columns.tolist())

customers.head()

In [ ]:
# Cell 3 — Apply cleaning and enrichment transformations
orders["order_date"] = pd.to_datetime(orders["order_date"])
feedback["feedback_date"] = pd.to_datetime(feedback["feedback_date"])

# Remove duplicate records and normalize string fields
customers = customers.drop_duplicates(subset=["customer_id"]).copy()
products = products.drop_duplicates(subset=["product_id"]).copy()
orders = orders.drop_duplicates(subset=["order_id"]).copy()
feedback = feedback.drop_duplicates(subset=["feedback_id"]).copy()

customers["country"] = customers["country"].str.strip().str.title()
products["product_name"] = products["product_name"].str.strip().str.title()
products["category"] = products["category"].str.strip().str.title()

# Join orders with customers and products for richer analytics
orders_enriched = orders.merge(customers, on="customer_id", how="left")
orders_enriched = orders_enriched.merge(products, on="product_id", how="left")
orders_enriched["sales_value"] = orders_enriched["quantity"] * orders_enriched["price"]
orders_enriched["order_month"] = orders_enriched["order_date"].dt.to_period("M").astype(str)

# Combine feedback with sentiment labels for customer insight
feedback_clean = feedback.merge(sentiments, on="feedback_id", how="left")
feedback_clean["sentiment_label"] = feedback_clean["sentiment"].fillna("unknown")
feedback_clean["feedback_bucket"] = feedback_clean["rating"].apply(
    lambda value: "positive" if value >= 4 else "neutral" if value == 3 else "negative"
)

print("Transformation complete")
orders_enriched[['order_id', 'customer_id', 'product_id', 'sales_value', 'order_month']].head()

In [ ]:
# Cell 4 — Calculate KPIs and save processed outputs
kpis = pd.DataFrame({
    "total_orders": [len(orders_enriched)],
    "total_revenue": [round(orders_enriched["sales_value"].sum(), 2)],
    "avg_order_value": [round(orders_enriched["sales_value"].mean(), 2)],
    "avg_rating": [round(feedback_clean["rating"].mean(), 2)],
    "positive_feedback_count": [int((feedback_clean["feedback_bucket"] == "positive").sum())],
    "negative_feedback_count": [int((feedback_clean["feedback_bucket"] == "negative").sum())],
    "distinct_customers": [customers["customer_id"].nunique()],
    "distinct_products": [products["product_id"].nunique()],
})

orders_enriched.to_csv(OUTPUT_DIR / "orders_enriched.csv", index=False)
feedback_clean.to_csv(OUTPUT_DIR / "feedback_clean.csv", index=False)
kpis.to_csv(OUTPUT_DIR / "kpis.csv", index=False)

print("Processed files written to:", OUTPUT_DIR)
kpis